# 04 — Optimisation Python

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- appliquer la hiérarchie d'optimisation (algorithme > structure > code) ;
- choisir la bonne structure de données pour chaque problème ;
- utiliser `functools.lru_cache` et `functools.cache` efficacement ;
- optimiser les boucles et les appels de fonctions ;
- appliquer des patterns d'optimisation éprouvés en Python pur.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- le profiling CPU et mémoire (notebooks 01 et 02) ;
- le benchmarking avec `timeit` et `pyperf` (notebook 03) ;
- les structures de données standard et leur complexité ;
- les décorateurs et les compréhensions.

## Plan

1. La hiérarchie d'optimisation
2. Choisir la bonne structure de données
3. Caching avec `functools`
4. Optimiser les boucles
5. Optimiser les chaînes
6. Optimiser les appels de fonctions
7. Optimiser les I/O
8. Patterns avancés
9. Synthèse
10. Exercices
11. Ressources

---

## 1. La hiérarchie d'optimisation

Optimisez toujours **dans cet ordre** :

| Priorité | Niveau | Gain typique | Exemple |
|---|---|---|---|
| 1 | **Algorithme** | 10x – 1000x | O(n^2) → O(n log n) |
| 2 | **Structure de données** | 2x – 100x | list → set pour `in` |
| 3 | **Code Python** | 1.2x – 5x | Boucle → compréhension |
| 4 | **Extension C** | 2x – 50x | Cython, Numba (notebook suivant) |
| 5 | **Parallélisme** | 2x – Nx | multiprocessing, asyncio |

**Ne descendez au niveau suivant que si le précédent ne suffit pas.**

### Exemple : l'algorithme fait toute la différence

In [ ]:
import timeit

def doublons_naif(lst):
    """O(n^2) — compare chaque paire"""
    doublons = []
    for i in range(len(lst)):
        for j in range(i + 1, len(lst)):
            if lst[i] == lst[j] and lst[i] not in doublons:
                doublons.append(lst[i])
    return doublons

def doublons_set(lst):
    """O(n) — utilise un ensemble"""
    vus = set()
    doublons = set()
    for x in lst:
        if x in vus:
            doublons.add(x)
        vus.add(x)
    return list(doublons)

In [ ]:
import random
data = random.choices(range(500), k=5000)

t_naif = timeit.timeit(lambda: doublons_naif(data), number=1)
t_set = timeit.timeit(lambda: doublons_set(data), number=1)
print(f"Naïf (O(n²)) : {t_naif:.3f}s")
print(f"Set  (O(n))  : {t_set:.6f}s")
print(f"Ratio : {t_naif / t_set:.0f}x")

---

## 2. Choisir la bonne structure de données

| Opération | `list` | `set` | `dict` | `deque` |
|---|---|---|---|---|
| `x in ...` | O(n) | **O(1)** | **O(1)** | O(n) |
| `append` | O(1)* | — | — | O(1) |
| `insert(0, x)` | O(n) | — | — | **O(1)** |
| `pop(0)` | O(n) | — | — | **O(1)** |
| Accès par index | **O(1)** | — | — | O(n) |
| Tri | O(n log n) | — | — | — |

*amorti

### `set` pour les tests d'appartenance

In [ ]:
N = 100_000
lst = list(range(N))
st = set(range(N))

# Chercher un élément absent (pire cas pour list)
cible = -1

t_list = timeit.timeit(lambda: cible in lst, number=100)
t_set = timeit.timeit(lambda: cible in st, number=100)
print(f"list : {t_list:.4f}s")
print(f"set  : {t_set:.6f}s")
print(f"set est {t_list / t_set:.0f}x plus rapide")

### `dict` comme index

In [ ]:
# Trouver un utilisateur par email dans une liste
users_list = [{"email": f"user{i}@example.com", "name": f"User {i}"} for i in range(10_000)]

# Version lente : recherche linéaire
def find_list(email):
    for u in users_list:
        if u["email"] == email:
            return u
    return None

# Version rapide : indexation par dict
users_index = {u["email"]: u for u in users_list}

def find_dict(email):
    return users_index.get(email)

cible = "user9999@example.com"
t_list = timeit.timeit(lambda: find_list(cible), number=1000)
t_dict = timeit.timeit(lambda: find_dict(cible), number=1000)
print(f"Recherche list : {t_list:.4f}s")
print(f"Recherche dict : {t_dict:.6f}s")
print(f"Dict est {t_list / t_dict:.0f}x plus rapide")

### `collections.deque` pour les files

In [ ]:
from collections import deque

def file_list(n):
    q = []
    for i in range(n):
        q.append(i)
    for _ in range(n):
        q.pop(0)  # O(n) à chaque pop !

def file_deque(n):
    q = deque()
    for i in range(n):
        q.append(i)
    for _ in range(n):
        q.popleft()  # O(1)

for n in [1_000, 10_000]:
    t_list = timeit.timeit(lambda: file_list(n), number=10)
    t_deque = timeit.timeit(lambda: file_deque(n), number=10)
    print(f"n={n:>6d}  list: {t_list:.4f}s  deque: {t_deque:.4f}s  ratio: {t_list/t_deque:.0f}x")

### `array.array` pour les tableaux numériques homogènes

In [ ]:
import sys
from array import array

lst = list(range(100_000))
arr = array('l', range(100_000))

print(f"list  : {sys.getsizeof(lst) / 1024:.0f} Ko")
print(f"array : {sys.getsizeof(arr) / 1024:.0f} Ko")
print(f"Ratio : {sys.getsizeof(lst) / sys.getsizeof(arr):.1f}x")

---

## 3. Caching avec `functools`

### `functools.lru_cache` — cache LRU borné

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

In [ ]:
# Sans cache : O(2^n) ; avec cache : O(n)
result = fibonacci(100)
print(f"fib(100) = {result}")
print(f"Cache info : {fibonacci.cache_info()}")

### `functools.cache` — cache illimité (Python 3.9+)

In [ ]:
from functools import cache

@cache
def factorielle(n):
    if n <= 1:
        return 1
    return n * factorielle(n - 1)

print(f"20! = {factorielle(20)}")
print(f"Cache info : {factorielle.cache_info()}")

### Quand utiliser le cache ?

| Situation | Cache utile ? |
|---|---|
| Fonction pure (même args → même résultat) | Oui |
| Fonction avec effets de bord | Non |
| Arguments hashables (int, str, tuple) | Oui |
| Arguments non hashables (list, dict) | Non (utiliser une clé) |
| Résultats très volumineux | Attention à la mémoire |

### Vider le cache

In [ ]:
fibonacci.cache_clear()
print(f"Après clear : {fibonacci.cache_info()}")

### Cache personnalisé avec TTL

In [ ]:
import time
from functools import wraps

def cache_ttl(seconds: float):
    def decorator(fn):
        _cache = {}
        @wraps(fn)
        def wrapper(*args):
            now = time.monotonic()
            if args in _cache:
                result, timestamp = _cache[args]
                if now - timestamp < seconds:
                    return result
            result = fn(*args)
            _cache[args] = (result, now)
            return result
        wrapper.cache = _cache
        return wrapper
    return decorator

@cache_ttl(seconds=2.0)
def donnee_couteuse(x):
    time.sleep(0.1)  # simule un appel lent
    return x * 42

print(donnee_couteuse(5))  # calcul
print(donnee_couteuse(5))  # cache
print(f"Taille cache : {len(donnee_couteuse.cache)}")

---

## 4. Optimiser les boucles

### 4.1. Compréhension vs boucle

In [ ]:
N = 100_000

t_boucle = timeit.timeit("""
result = []
for i in range(N):
    result.append(i ** 2)
""", globals={"N": N}, number=100)

t_comp = timeit.timeit("[i ** 2 for i in range(N)]", globals={"N": N}, number=100)

print(f"Boucle + append : {t_boucle:.3f}s")
print(f"Compréhension   : {t_comp:.3f}s")
print(f"Ratio : {t_boucle / t_comp:.1f}x")

Les compréhensions sont plus rapides car :
1. Moins de bytecode (pas de `LOAD_ATTR` pour `.append`) ;
2. Optimisées en C depuis PEP 709 (Python 3.12+).

### 4.2. Éviter les lookups répétés

In [ ]:
import math

# LENT : math.sqrt est résolu à chaque itération
def somme_racines_lent(n):
    total = 0
    for i in range(1, n):
        total += math.sqrt(i)
    return total

# RAPIDE : variable locale
def somme_racines_rapide(n):
    sqrt = math.sqrt  # binding local
    total = 0
    for i in range(1, n):
        total += sqrt(i)
    return total

t_lent = timeit.timeit(lambda: somme_racines_lent(100_000), number=10)
t_rapide = timeit.timeit(lambda: somme_racines_rapide(100_000), number=10)
print(f"Avec lookup  : {t_lent:.3f}s")
print(f"Variable loc : {t_rapide:.3f}s")
print(f"Gain : {(1 - t_rapide / t_lent) * 100:.0f}%")

### 4.3. `map` et `filter` vs compréhensions

In [ ]:
data = list(range(100_000))

t_comp = timeit.timeit(lambda: [x * 2 for x in data], number=100)
t_map = timeit.timeit(lambda: list(map(lambda x: x * 2, data)), number=100)

print(f"Compréhension : {t_comp:.3f}s")
print(f"map + lambda  : {t_map:.3f}s")
# map est souvent plus lent avec lambda (surcoût d'appel Python)

### 4.4. Utiliser les builtins C

In [ ]:
# sum() en C vs boucle Python
def somme_boucle(n):
    total = 0
    for i in range(n):
        total += i
    return total

t_boucle = timeit.timeit(lambda: somme_boucle(1_000_000), number=10)
t_sum = timeit.timeit(lambda: sum(range(1_000_000)), number=10)
print(f"Boucle  : {t_boucle:.3f}s")
print(f"sum()   : {t_sum:.3f}s")
print(f"sum() est {t_boucle / t_sum:.1f}x plus rapide")

---

## 5. Optimiser les chaînes

### `str.join` vs concaténation

In [ ]:
mots = [f"mot{i}" for i in range(10_000)]

def concat_plus(mots):
    result = ""
    for m in mots:
        result += m + " "
    return result

def concat_join(mots):
    return " ".join(mots)

t_plus = timeit.timeit(lambda: concat_plus(mots), number=100)
t_join = timeit.timeit(lambda: concat_join(mots), number=100)
print(f"+= : {t_plus:.4f}s")
print(f"join : {t_join:.4f}s")
print(f"join est {t_plus / t_join:.0f}x plus rapide")

### f-strings vs `format()` vs `%`

In [ ]:
nom, age = "Alice", 30

t_fstring = timeit.timeit(lambda: f"{nom} a {age} ans", number=1_000_000)
t_format = timeit.timeit(lambda: "{} a {} ans".format(nom, age), number=1_000_000)
t_pourcent = timeit.timeit(lambda: "%s a %d ans" % (nom, age), number=1_000_000)

print(f"f-string : {t_fstring:.3f}s")
print(f"format() : {t_format:.3f}s")
print(f"%%        : {t_pourcent:.3f}s")

---

## 6. Optimiser les appels de fonctions

### L'appel de fonction a un coût

In [ ]:
def identite(x):
    return x

t_direct = timeit.timeit("x = 42", number=10_000_000)
t_appel = timeit.timeit("identite(42)", globals={"identite": identite}, number=10_000_000)
print(f"Affectation directe : {t_direct:.3f}s")
print(f"Appel de fonction   : {t_appel:.3f}s")
print(f"Coût d'un appel : ~{(t_appel - t_direct) / 10_000_000 * 1e9:.0f} ns")

### `__slots__` pour les petits objets fréquents

In [ ]:
class PointNormal:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class PointSlots:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x = x
        self.y = y

t_normal = timeit.timeit(lambda: PointNormal(1.0, 2.0), number=1_000_000)
t_slots = timeit.timeit(lambda: PointSlots(1.0, 2.0), number=1_000_000)
print(f"Normal   : {t_normal:.3f}s")
print(f"__slots__: {t_slots:.3f}s")

### Éviter `*args/**kwargs` inutiles

In [ ]:
def avec_kwargs(**kwargs):
    return kwargs.get("x", 0) + kwargs.get("y", 0)

def sans_kwargs(x=0, y=0):
    return x + y

t_kwargs = timeit.timeit(lambda: avec_kwargs(x=1, y=2), number=1_000_000)
t_normal = timeit.timeit(lambda: sans_kwargs(x=1, y=2), number=1_000_000)
print(f"**kwargs : {t_kwargs:.3f}s")
print(f"Params   : {t_normal:.3f}s")

---

## 7. Optimiser les I/O

### Lire par blocs

In [ ]:
import tempfile
import os

# Créer un fichier de test
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    tmp_path = f.name
    for i in range(100_000):
        f.write(f"ligne numéro {i}\n")

# Lecture ligne par ligne vs readlines vs read
t_iter = timeit.timeit(lambda: sum(1 for _ in open(tmp_path)), number=10)
t_readlines = timeit.timeit(lambda: len(open(tmp_path).readlines()), number=10)
t_read = timeit.timeit(lambda: open(tmp_path).read().count("\n"), number=10)

print(f"Itérateur  : {t_iter:.3f}s")
print(f"readlines(): {t_readlines:.3f}s")
print(f"read()     : {t_read:.3f}s")

os.unlink(tmp_path)

### Utiliser `io.BufferedWriter` avec un gros buffer

In [ ]:
import io

with tempfile.NamedTemporaryFile(delete=False) as f:
    tmp_path = f.name

# Buffer par défaut (8 Ko)
def ecrire_defaut():
    with open(tmp_path, "wb") as f:
        for i in range(100_000):
            f.write(b"x" * 100)

# Gros buffer (1 Mo)
def ecrire_gros_buffer():
    with open(tmp_path, "wb", buffering=1_048_576) as f:
        for i in range(100_000):
            f.write(b"x" * 100)

t_defaut = timeit.timeit(ecrire_defaut, number=5)
t_gros = timeit.timeit(ecrire_gros_buffer, number=5)
print(f"Buffer 8Ko  : {t_defaut:.3f}s")
print(f"Buffer 1Mo  : {t_gros:.3f}s")

os.unlink(tmp_path)

---

## 8. Patterns avancés

### 8.1. Sentinel pattern pour éviter `if key in dict`

In [ ]:
_SENTINEL = object()

def get_or_compute(cache, key, compute_fn):
    val = cache.get(key, _SENTINEL)
    if val is _SENTINEL:
        val = compute_fn(key)
        cache[key] = val
    return val

# Un seul lookup dict au lieu de deux (in + get)

### 8.2. `operator` au lieu de lambdas

In [ ]:
import operator
from functools import reduce

data = list(range(1, 1001))

t_lambda = timeit.timeit(lambda: reduce(lambda a, b: a + b, data), number=10_000)
t_op = timeit.timeit(lambda: reduce(operator.add, data), number=10_000)
print(f"lambda : {t_lambda:.3f}s")
print(f"operator.add : {t_op:.3f}s")

### 8.3. `itertools` pour les pipelines mémoire-efficaces

In [ ]:
import itertools

# Traiter un fichier de 1M lignes sans tout charger en mémoire
def pipeline_itertools(iterable, n_premiers=100):
    filtré = (x for x in iterable if x % 3 == 0)
    transformé = (x ** 2 for x in filtré)
    return list(itertools.islice(transformé, n_premiers))

# Fonctionne sur n'importe quel itérable, même infini
result = pipeline_itertools(range(1_000_000))
print(f"Premiers résultats : {result[:5]}...")

### 8.4. `__slots__` + `__match_args__` pour les records

In [ ]:
class Evenement:
    __slots__ = ("type_", "timestamp", "payload")
    __match_args__ = ("type_", "timestamp", "payload")

    def __init__(self, type_: str, timestamp: float, payload: dict):
        self.type_ = type_
        self.timestamp = timestamp
        self.payload = payload

    def __repr__(self):
        return f"Evenement({self.type_!r}, {self.timestamp}, {self.payload})"

evt = Evenement("click", 1234567890.0, {"x": 100, "y": 200})
print(evt)
print(f"Taille : {sys.getsizeof(evt)} octets")

---

## 9. Synthèse

| Technique | Gain typique | Quand l'utiliser |
|---|---|---|
| Meilleur algorithme | 10-1000x | Toujours en premier |
| `set`/`dict` au lieu de `list` pour `in` | 100-10000x | Recherche fréquente |
| `lru_cache` | Jusqu'à exponentiel | Fonctions pures, args hashables |
| Compréhension vs boucle | 1.2-2x | Création de listes |
| Binding local | 5-15% | Boucles critiques |
| `str.join` vs `+=` | 5-100x | Concaténation de chaînes |
| Builtins C (`sum`, `sorted`) | 2-5x | Opérations standard |
| `__slots__` | 30-50% mémoire | Petits objets fréquents |

**Règles à retenir :**
- Profilez avant d'optimiser — ne devinez jamais.
- Un bon algorithme bat toujours un mauvais algorithme optimisé.
- Les micro-optimisations Python ne servent que dans les **boucles critiques**.
- `lru_cache` est la technique la plus facile à appliquer pour un gain immédiat.

---

## 10. Exercices

### Exercice 1 — Optimiser une recherche *(facile)*

Le code suivant vérifie si des emails sont dans une liste noire. Optimisez-le :

```python
blacklist = ["spam@evil.com", "phish@bad.org", ...]  # 10 000 entrées
emails = [...]  # 100 000 emails à vérifier

bloqués = []
for email in emails:
    if email in blacklist:  # O(n) à chaque vérification !
        bloqués.append(email)
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Optimisation", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import random

blacklist = [f"spam{i}@evil.com" for i in range(10_000)]
emails = [f"user{i}@example.com" for i in range(95_000)] + random.sample(blacklist, 5_000)
random.shuffle(emails)

# Version optimisée : convertir en set
blacklist_set = set(blacklist)
bloqués = [email for email in emails if email in blacklist_set]

print(f"Emails bloqués : {len(bloqués)}")
```

</details>

### Exercice 2 — Fibonacci avec cache *(facile)*

Comparez le temps de calcul de `fibonacci(35)` avec et sans `lru_cache`. Mesurez avec `timeit`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Optimisation", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import timeit
from functools import lru_cache

def fib_naif(n):
    if n < 2:
        return n
    return fib_naif(n - 1) + fib_naif(n - 2)

@lru_cache(maxsize=None)
def fib_cache(n):
    if n < 2:
        return n
    return fib_cache(n - 1) + fib_cache(n - 2)

t_naif = timeit.timeit(lambda: fib_naif(30), number=1)
fib_cache.cache_clear()
t_cache = timeit.timeit(lambda: fib_cache(30), number=1)

print(f"Sans cache : {t_naif:.3f}s")
print(f"Avec cache : {t_cache:.6f}s")
print(f"Ratio : {t_naif / t_cache:.0f}x")
```

</details>

### Exercice 3 — Optimiser un pipeline de données *(moyen)*

Optimisez le code suivant qui traite une liste de transactions :

```python
def analyser_transactions(transactions):
    # Filtrer les montants > 1000
    gros = []
    for t in transactions:
        if t["montant"] > 1000:
            gros.append(t)

    # Trier par date
    gros_tries = sorted(gros, key=lambda t: t["date"])

    # Extraire les emails uniques
    emails = []
    for t in gros_tries:
        if t["email"] not in emails:
            emails.append(t["email"])

    return emails
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Optimisation", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
def analyser_transactions_v2(transactions):
    # Compréhension + tri en un seul passage
    gros_tries = sorted(
        (t for t in transactions if t["montant"] > 1000),
        key=lambda t: t["date"]
    )
    # dict.fromkeys préserve l'ordre et déduplique en O(1) par lookup
    return list(dict.fromkeys(t["email"] for t in gros_tries))

# La version originale utilise `email not in emails` qui est O(n)
# pour chaque vérification dans une liste.
# dict.fromkeys est O(1) par insertion.
```

</details>

### Exercice 4 — Cache LRU pour une API *(moyen)*

Implémenter un cache LRU **avec TTL** (Time To Live) sans utiliser `functools.lru_cache`. Votre cache doit :
1. Stocker au maximum `maxsize` entrées ;
2. Évincer la plus anciennement accédée quand le cache est plein ;
3. Invalider les entrées plus vieilles que `ttl` secondes.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Optimisation", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import time
from collections import OrderedDict

class LRUCache:
    def __init__(self, maxsize: int = 128, ttl: float = 60.0):
        self.maxsize = maxsize
        self.ttl = ttl
        self._cache = OrderedDict()

    def get(self, key):
        if key in self._cache:
            value, timestamp = self._cache[key]
            if time.monotonic() - timestamp < self.ttl:
                self._cache.move_to_end(key)
                return value
            else:
                del self._cache[key]
        return None

    def put(self, key, value):
        if key in self._cache:
            del self._cache[key]
        elif len(self._cache) >= self.maxsize:
            self._cache.popitem(last=False)
        self._cache[key] = (value, time.monotonic())

cache = LRUCache(maxsize=3, ttl=5.0)
cache.put("a", 1)
cache.put("b", 2)
cache.put("c", 3)
print(cache.get("a"))  # 1 (a est désormais le plus récent)
cache.put("d", 4)       # b est évincé
print(cache.get("b"))   # None
```

</details>

### Exercice 5 — Benchmark comparatif complet *(difficile)*

Écrire un rapport de benchmark complet qui compare 4 façons de compter les mots dans un texte :
1. Boucle `for` avec `dict` ;
2. `collections.Counter` ;
3. `defaultdict(int)` ;
4. `dict.setdefault`.

Pour chaque approche, mesurez avec `timeit` sur 3 tailles de texte (1K, 10K, 100K mots). Affichez un tableau formaté.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Optimisation", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
import timeit
from collections import Counter, defaultdict
import random

def compter_dict(mots):
    d = {}
    for m in mots:
        if m in d:
            d[m] += 1
        else:
            d[m] = 1
    return d

def compter_counter(mots):
    return Counter(mots)

def compter_defaultdict(mots):
    d = defaultdict(int)
    for m in mots:
        d[m] += 1
    return dict(d)

def compter_setdefault(mots):
    d = {}
    for m in mots:
        d[m] = d.get(m, 0) + 1
    return d

vocabulaire = [f"mot{i}" for i in range(500)]

print(f"{'Méthode':<20s} {'1K (µs)':>10s} {'10K (µs)':>10s} {'100K (µs)':>10s}")
print("-" * 55)

for nom, fn in [("dict", compter_dict), ("Counter", compter_counter),
                ("defaultdict", compter_defaultdict), ("setdefault", compter_setdefault)]:
    temps = []
    for n in [1_000, 10_000, 100_000]:
        mots = random.choices(vocabulaire, k=n)
        t = min(timeit.repeat(lambda: fn(mots), number=100, repeat=3))
        temps.append(f"{t / 100 * 1e6:.0f}")
    print(f"{nom:<20s} {temps[0]:>10s} {temps[1]:>10s} {temps[2]:>10s}")
```

</details>

---

## 11. Ressources

- [Module `functools` — `lru_cache`](https://docs.python.org/3/library/functools.html#functools.lru_cache)
- [TimeComplexity — Python Wiki](https://wiki.python.org/moin/TimeComplexity)
- [High Performance Python — Micha Gorelick & Ian Ozsvald](https://www.oreilly.com/library/view/high-performance-python/9781492055013/)
- [Python Speed — tips](https://wiki.python.org/moin/PythonSpeed/PerformanceTips)